# Audit the position-coded nodal SDS archive

This notebook is the preflight step between:

```text
60_downsample_GP_to_DP_in_position_sds.ipynb
```

and:

```text
70_compute_nodal_rsam_from_position_coded_sds.ipynb
```

It audits:

```text
/Volumes/tachyon/LBSSP_DATA/nodal_sds_position_codes
```

using FLOVOpy's `EnhancedSDSClient`.

The audit answers four distinct questions:

1. Which `NET.STA.LOC.CHA` identifiers are present?
2. How much data are present for each identifier on each UTC day?
3. Are expected components missing at any position/deployment?
4. Are there station, location, or channel codes that do not match the
   position-coded archive conventions?

The notebook writes wide, long, and summary availability tables, a heatmap,
component-completeness tables, and a compact audit summary that stage 70 can
read as a pre-processing warning.

Daily availability is measured over complete UTC days. Partial deployment days
can therefore have legitimately low percentages; the notebook reports them but
does not automatically classify them as failed.

In [ ]:
from __future__ import annotations

from fnmatch import fnmatch
from pathlib import Path
from typing import Iterable, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from obspy import UTCDateTime

from flovopy.enhanced.sdsclient import EnhancedSDSClient

## Configuration

`END` is exclusive in the daily availability loop. Use `2026-05-21` to include
all of 20 May.

The expected location codes follow the current position-coded archive:

- T1: `N1`, `N2`, `N3`
- T3: `N4`

Step 60 standardized the nodal data to `DPE`, `DPN`, and `DPZ`.

In [ ]:
SDS_ROOT = Path(
    "/Volumes/tachyon/LBSSP_DATA/nodal_sds_position_codes"
)

QC_ROOT = Path(
    "/Volumes/tachyon/LBSSP_DATA/nodal_qc_position_codes"
)
QC_ROOT.mkdir(parents=True, exist_ok=True)

START = UTCDateTime("2026-05-16T00:00:00")
END = UTCDateTime("2026-05-21T00:00:00")

NETWORK_LOCATIONS = {
    "T1": ["N1", "N2", "N3"],
    "T3": ["N4"],
}

NETWORKS = list(NETWORK_LOCATIONS)
STATION_PATTERNS = ["*"]
LOCATION_PATTERNS = sorted(
    {
        location
        for locations in NETWORK_LOCATIONS.values()
        for location in locations
    }
)
CHANNELS = ["DPE", "DPN", "DPZ"]

SKIP_LOW_RATE_CHANNELS = False
SHOW_PROGRESS = True
VERBOSE = False

# These thresholds are descriptive flags, not automatic data rejection.
LOW_DAILY_AVAILABILITY_THRESHOLD = 0.90
VERY_LOW_DAILY_AVAILABILITY_THRESHOLD = 0.50

OUT_PREFIX = QC_ROOT / "position_coded_nodal_sds_availability"

WIDE_CSV = OUT_PREFIX.with_name(
    OUT_PREFIX.name + "_wide.csv"
)
LONG_CSV = OUT_PREFIX.with_name(
    OUT_PREFIX.name + "_long.csv"
)
SUMMARY_CSV = OUT_PREFIX.with_name(
    OUT_PREFIX.name + "_summary.csv"
)
LOW_AVAILABILITY_CSV = OUT_PREFIX.with_name(
    OUT_PREFIX.name + "_low_availability.csv"
)
COMPONENT_COMPLETENESS_CSV = OUT_PREFIX.with_name(
    OUT_PREFIX.name + "_component_completeness.csv"
)
CODE_QC_CSV = OUT_PREFIX.with_name(
    OUT_PREFIX.name + "_code_qc.csv"
)
HEATMAP_PNG = OUT_PREFIX.with_name(
    OUT_PREFIX.name + "_heatmap.png"
)
AUDIT_SUMMARY_CSV = (
    QC_ROOT / "position_coded_sds_audit_summary.csv"
)

print(f"SDS root: {SDS_ROOT}")
print(f"QC root:  {QC_ROOT}")
print(f"Audit:    {START} to {END} (end exclusive)")

## Helper functions

In [ ]:
def seed_id_matches(
    seed_id: str,
    networks: Sequence[str],
    stations: Sequence[str],
    locations: Sequence[str],
    channels: Sequence[str],
) -> bool:
    """Return True when a SEED id matches all selector groups."""
    parts = seed_id.split(".")

    if len(parts) != 4:
        return False

    network, station, location, channel = parts

    def any_match(
        value: str,
        patterns: Sequence[str],
    ) -> bool:
        return any(
            fnmatch(value, pattern)
            for pattern in patterns
        )

    return (
        any_match(network, networks)
        and any_match(station, stations)
        and any_match(location, locations)
        and any_match(channel, channels)
    )


def filter_trace_ids(
    trace_ids: Iterable[str],
    *,
    networks: Sequence[str],
    stations: Sequence[str],
    locations: Sequence[str],
    channels: Sequence[str],
) -> list[str]:
    """Filter discovered SEED ids with shell-style wildcards."""
    return sorted(
        trace_id
        for trace_id in trace_ids
        if seed_id_matches(
            trace_id,
            networks,
            stations,
            locations,
            channels,
        )
    )


def parse_seed_id(
    seed_id: str,
) -> tuple[str, str, str, str]:
    parts = seed_id.split(".")

    if len(parts) != 4:
        raise ValueError(
            f"Expected NET.STA.LOC.CHA, got {seed_id!r}"
        )

    return tuple(parts)


def station_code_to_position_m(
    station: str,
) -> float:
    """Interpret integer station code as centimetres along profile."""
    text = str(station).strip()

    if text.endswith(".0"):
        text = text[:-2]

    if not text.isdigit():
        raise ValueError(
            f"Station code {station!r} is not an integer centimetre position."
        )

    return int(text) / 100.0


def wide_to_long(
    wide: pd.DataFrame,
) -> pd.DataFrame:
    """Convert daily availability from wide to tidy long format."""
    if wide.empty:
        return pd.DataFrame(
            columns=[
                "date",
                "network",
                "station",
                "location",
                "channel",
                "seed_id",
                "availability",
                "availability_percent",
            ]
        )

    long = wide.melt(
        id_vars=["date"],
        var_name="seed_id",
        value_name="availability",
    )

    long["availability"] = pd.to_numeric(
        long["availability"],
        errors="coerce",
    )
    long["availability_percent"] = (
        100.0 * long["availability"]
    )

    parts = long["seed_id"].str.split(
        ".",
        expand=True,
    )

    if parts.shape[1] == 4:
        long.insert(1, "network", parts[0])
        long.insert(2, "station", parts[1])
        long.insert(3, "location", parts[2])
        long.insert(4, "channel", parts[3])

    return long


def summarize_long(
    long: pd.DataFrame,
) -> pd.DataFrame:
    """Summarize daily availability for each SEED id."""
    if long.empty:
        return pd.DataFrame()

    summary = (
        long
        .groupby("seed_id", as_index=False)
        .agg(
            network=("network", "first"),
            station=("station", "first"),
            location=("location", "first"),
            channel=("channel", "first"),
            n_days=("availability", "size"),
            n_days_with_data=(
                "availability",
                lambda values: int((values > 0).sum()),
            ),
            n_days_at_or_above_90_percent=(
                "availability",
                lambda values: int(
                    (
                        values
                        >= LOW_DAILY_AVAILABILITY_THRESHOLD
                    ).sum()
                ),
            ),
            mean_availability=("availability", "mean"),
            median_availability=("availability", "median"),
            min_availability=("availability", "min"),
            max_availability=("availability", "max"),
        )
    )

    for column in [
        "mean_availability",
        "median_availability",
        "min_availability",
        "max_availability",
    ]:
        summary[column + "_percent"] = (
            100.0 * summary[column]
        )

    summary["position_m"] = summary["station"].map(
        station_code_to_position_m
    )

    return summary.sort_values(
        [
            "network",
            "location",
            "position_m",
            "channel",
        ],
        kind="stable",
    ).reset_index(drop=True)

## Discover trace identifiers

Discovery is performed across the complete audit interval. The resulting list
is filtered to the expected networks, deployment location codes, and DP
components.

In [ ]:
client = EnhancedSDSClient(SDS_ROOT)

all_discovered_trace_ids = sorted(
    client.iter_trace_ids(
        START,
        END,
        skip_low_rate=SKIP_LOW_RATE_CHANNELS,
    )
)

trace_ids = filter_trace_ids(
    all_discovered_trace_ids,
    networks=NETWORKS,
    stations=STATION_PATTERNS,
    locations=LOCATION_PATTERNS,
    channels=CHANNELS,
)

print(
    f"Discovered {len(all_discovered_trace_ids)} total trace IDs "
    f"between {START} and {END}."
)
print(
    f"Retained {len(trace_ids)} expected position-coded nodal IDs."
)

for trace_id in trace_ids[:30]:
    print(" ", trace_id)

if len(trace_ids) > 30:
    print(f"  ... {len(trace_ids) - 30} more")

if not trace_ids:
    raise RuntimeError(
        "No trace IDs matched the configured network, location, "
        "station, and channel selectors."
    )

## Audit SEED-code conventions

This table includes every discovered identifier, including identifiers excluded
from the availability calculation. It therefore reveals unexpected channels,
locations, networks, or non-numeric station codes.

In [ ]:
code_qc_rows = []

for seed_id in all_discovered_trace_ids:
    try:
        network, station, location, channel = parse_seed_id(
            seed_id
        )
    except ValueError as exc:
        code_qc_rows.append(
            {
                "seed_id": seed_id,
                "network": None,
                "station": None,
                "location": None,
                "channel": None,
                "position_m": np.nan,
                "network_expected": False,
                "location_expected_for_network": False,
                "channel_expected": False,
                "station_is_position_code": False,
                "included_in_availability_audit": False,
                "issue": str(exc),
            }
        )
        continue

    try:
        position_m = station_code_to_position_m(
            station
        )
        station_ok = True
    except ValueError:
        position_m = np.nan
        station_ok = False

    network_ok = network in NETWORK_LOCATIONS
    location_ok = (
        network_ok
        and location in NETWORK_LOCATIONS[network]
    )
    channel_ok = channel in CHANNELS
    included = seed_id in trace_ids

    issues = []

    if not network_ok:
        issues.append("unexpected network")
    if not location_ok:
        issues.append("unexpected location for network")
    if not channel_ok:
        issues.append("unexpected channel")
    if not station_ok:
        issues.append("station is not centimetre position code")

    code_qc_rows.append(
        {
            "seed_id": seed_id,
            "network": network,
            "station": station,
            "location": location,
            "channel": channel,
            "position_m": position_m,
            "network_expected": network_ok,
            "location_expected_for_network": location_ok,
            "channel_expected": channel_ok,
            "station_is_position_code": station_ok,
            "included_in_availability_audit": included,
            "issue": "; ".join(issues),
        }
    )

code_qc = pd.DataFrame(code_qc_rows).sort_values(
    [
        "included_in_availability_audit",
        "network",
        "location",
        "position_m",
        "channel",
    ],
    ascending=[True, True, True, True, True],
    na_position="last",
).reset_index(drop=True)

code_qc.to_csv(
    CODE_QC_CSV,
    index=False,
)

unexpected_code_rows = code_qc[
    code_qc["issue"].str.len() > 0
]

print(f"Wrote {CODE_QC_CSV}")
print(
    f"Unexpected or malformed identifiers: "
    f"{len(unexpected_code_rows)}"
)

display(unexpected_code_rows)

## Calculate daily archive availability

The FLOVOpy availability values range from 0 to 1. The start date is inclusive
and the end date is exclusive.

In [ ]:
wide, audited_trace_ids = client.get_availability(
    startday=START,
    endday=END,
    trace_ids=trace_ids,
    skip_low_rate_channels=SKIP_LOW_RATE_CHANNELS,
    progress=SHOW_PROGRESS,
    verbose=VERBOSE,
)

if wide.empty:
    raise RuntimeError(
        "FLOVOpy returned an empty availability table."
    )

wide.to_csv(
    WIDE_CSV,
    index=False,
)

long = wide_to_long(wide)
long.to_csv(
    LONG_CSV,
    index=False,
)

summary = summarize_long(long)
summary.to_csv(
    SUMMARY_CSV,
    index=False,
)

print(f"Wrote {WIDE_CSV}")
print(f"Wrote {LONG_CSV}")
print(f"Wrote {SUMMARY_CSV}")

display(wide.head())
display(summary.head())

## Identify low-availability day/trace combinations

This is a descriptive screening table. Low values on the first or last day of
a deployment may simply reflect deployment or retrieval partway through a UTC
day.

In [ ]:
low_availability = long[
    long["availability"]
    < LOW_DAILY_AVAILABILITY_THRESHOLD
].copy()

low_availability["severity"] = np.where(
    low_availability["availability"]
    < VERY_LOW_DAILY_AVAILABILITY_THRESHOLD,
    "very low",
    "below 90%",
)

low_availability["position_m"] = (
    low_availability["station"]
    .map(station_code_to_position_m)
)

low_availability = low_availability.sort_values(
    [
        "availability",
        "network",
        "location",
        "position_m",
        "channel",
        "date",
    ],
    kind="stable",
).reset_index(drop=True)

low_availability.to_csv(
    LOW_AVAILABILITY_CSV,
    index=False,
)

print(f"Wrote {LOW_AVAILABILITY_CSV}")
print(
    f"Rows below "
    f"{100 * LOW_DAILY_AVAILABILITY_THRESHOLD:.0f}%: "
    f"{len(low_availability)}"
)

display(
    low_availability[
        [
            "date",
            "seed_id",
            "position_m",
            "availability_percent",
            "severity",
        ]
    ].head(50)
)

## Check component completeness

For each network, deployment location, and station position, the notebook
checks whether all three expected channels (`DPE`, `DPN`, and `DPZ`) were
discovered at least once during the audit interval.

This is independent of daily availability: a component may exist but still
contain substantial gaps.

In [ ]:
component_rows = []

trace_metadata = pd.DataFrame(
    [
        dict(
            zip(
                [
                    "network",
                    "station",
                    "location",
                    "channel",
                ],
                parse_seed_id(seed_id),
            )
        )
        for seed_id in trace_ids
    ]
)

trace_metadata["position_m"] = (
    trace_metadata["station"]
    .map(station_code_to_position_m)
)

for (
    network,
    location,
    station,
    position_m,
), group in trace_metadata.groupby(
    [
        "network",
        "location",
        "station",
        "position_m",
    ],
    sort=True,
):
    present = sorted(
        set(group["channel"])
    )
    missing = sorted(
        set(CHANNELS) - set(present)
    )

    component_rows.append(
        {
            "network": network,
            "location": location,
            "station": station,
            "position_m": position_m,
            "channels_present": ", ".join(present),
            "n_channels_present": len(present),
            "missing_channels": ", ".join(missing),
            "complete_three_component": not missing,
        }
    )

component_completeness = pd.DataFrame(
    component_rows
).sort_values(
    [
        "network",
        "location",
        "position_m",
    ],
    kind="stable",
).reset_index(drop=True)

component_completeness.to_csv(
    COMPONENT_COMPLETENESS_CSV,
    index=False,
)

incomplete_components = component_completeness[
    ~component_completeness["complete_three_component"]
]

print(f"Wrote {COMPONENT_COMPLETENESS_CSV}")
print(
    f"Incomplete position/deployment combinations: "
    f"{len(incomplete_components)}"
)

display(incomplete_components)

## Availability summary by deployment and component

This condenses the trace-level results without hiding individual problem
records.

In [ ]:
deployment_summary = (
    summary
    .groupby(
        [
            "network",
            "location",
            "channel",
        ],
        as_index=False,
    )
    .agg(
        trace_ids=("seed_id", "nunique"),
        distinct_positions=("position_m", "nunique"),
        minimum_position_m=("position_m", "min"),
        maximum_position_m=("position_m", "max"),
        mean_availability_percent=(
            "mean_availability_percent",
            "mean",
        ),
        minimum_trace_mean_availability_percent=(
            "mean_availability_percent",
            "min",
        ),
        minimum_daily_availability_percent=(
            "min_availability_percent",
            "min",
        ),
    )
    .sort_values(
        [
            "network",
            "location",
            "channel",
        ]
    )
)

display(deployment_summary)

## Plot the daily availability heatmap

The first attempt uses FLOVOpy's built-in plotter, preserving the behaviour of
the original audit script. A fallback Matplotlib implementation is provided in
case the installed FLOVOpy version cannot plot the table.

In [ ]:
def plot_availability_fallback(
    wide: pd.DataFrame,
    *,
    outfile: Path,
) -> None:
    data = wide.copy()
    dates = pd.to_datetime(
        data.pop("date")
    )

    values = data.to_numpy(dtype=float).T
    trace_labels = list(data.columns)

    fig_height = max(
        6.0,
        0.18 * len(trace_labels),
    )

    fig, ax = plt.subplots(
        figsize=(12, fig_height),
        constrained_layout=True,
    )

    image = ax.imshow(
        values,
        aspect="auto",
        interpolation="nearest",
        vmin=0.0,
        vmax=1.0,
    )

    ax.set_xticks(
        np.arange(len(dates))
    )
    ax.set_xticklabels(
        [
            date.strftime("%Y-%m-%d")
            for date in dates
        ],
        rotation=45,
        ha="right",
    )

    ax.set_yticks(
        np.arange(len(trace_labels))
    )
    ax.set_yticklabels(
        trace_labels,
        fontsize=6,
    )

    ax.set_xlabel("UTC day")
    ax.set_ylabel("SEED identifier")
    ax.set_title(
        "Position-coded nodal SDS daily availability"
    )

    colorbar = fig.colorbar(
        image,
        ax=ax,
        pad=0.01,
    )
    colorbar.set_label(
        "Fractional daily availability"
    )

    fig.savefig(
        outfile,
        dpi=200,
        bbox_inches="tight",
    )
    plt.close(fig)


try:
    client.plot_availability(
        availability_df=wide,
        outfile=str(HEATMAP_PNG),
        progress=False,
        verbose=VERBOSE,
    )
    print(
        f"Wrote FLOVOpy heatmap: {HEATMAP_PNG}"
    )
except Exception as exc:
    print(
        "FLOVOpy availability plot failed; "
        f"using fallback plotter: {exc}"
    )
    plot_availability_fallback(
        wide,
        outfile=HEATMAP_PNG,
    )
    print(
        f"Wrote fallback heatmap: {HEATMAP_PNG}"
    )

## Compact audit summary for stage 70

This file is intentionally small. Stage 70 can read it at startup and display
the audit status before computing RSAM.

The status is:

- `PASS` when no malformed identifiers, missing components, or very-low daily
  availability records are found;
- `REVIEW` otherwise.

`REVIEW` is a warning, not an instruction to stop processing.

In [ ]:
very_low_records = long[
    long["availability"]
    < VERY_LOW_DAILY_AVAILABILITY_THRESHOLD
]

audit_status = (
    "PASS"
    if (
        len(unexpected_code_rows) == 0
        and len(incomplete_components) == 0
        and len(very_low_records) == 0
    )
    else "REVIEW"
)

audit_summary = pd.DataFrame(
    [
        {
            "audit_status": audit_status,
            "sds_root": str(SDS_ROOT),
            "start_utc": str(START),
            "end_utc_exclusive": str(END),
            "trace_ids_audited": len(trace_ids),
            "distinct_networks": (
                trace_metadata["network"].nunique()
            ),
            "distinct_deployment_locations": (
                trace_metadata[
                    [
                        "network",
                        "location",
                    ]
                ]
                .drop_duplicates()
                .shape[0]
            ),
            "distinct_position_codes": (
                trace_metadata["station"].nunique()
            ),
            "unexpected_or_malformed_ids": (
                len(unexpected_code_rows)
            ),
            "incomplete_three_component_positions": (
                len(incomplete_components)
            ),
            "daily_records_below_90_percent": (
                len(low_availability)
            ),
            "daily_records_below_50_percent": (
                len(very_low_records)
            ),
            "mean_availability_percent_all_records": (
                100.0 * long["availability"].mean()
            ),
            "minimum_daily_availability_percent": (
                100.0 * long["availability"].min()
            ),
            "wide_csv": str(WIDE_CSV),
            "long_csv": str(LONG_CSV),
            "summary_csv": str(SUMMARY_CSV),
            "component_completeness_csv": str(
                COMPONENT_COMPLETENESS_CSV
            ),
            "code_qc_csv": str(CODE_QC_CSV),
            "heatmap_png": str(HEATMAP_PNG),
        }
    ]
)

audit_summary.to_csv(
    AUDIT_SUMMARY_CSV,
    index=False,
)

print(f"Wrote {AUDIT_SUMMARY_CSV}")
display(audit_summary)

## Suggested stage-70 preflight cell

The following optional cell can be copied into
`70_compute_nodal_rsam_from_position_coded_sds.ipynb`:

```python
AUDIT_SUMMARY_FILE = (
    Path("/Volumes/tachyon/LBSSP_DATA/nodal_qc_position_codes")
    / "position_coded_sds_audit_summary.csv"
)

if AUDIT_SUMMARY_FILE.exists():
    audit_summary = pd.read_csv(AUDIT_SUMMARY_FILE)
    display(audit_summary)

    if (audit_summary["audit_status"] != "PASS").any():
        print(
            "The stage-65 audit returned REVIEW. "
            "RSAM processing may proceed, but inspect the audit outputs."
        )
else:
    print(
        "No stage-65 audit summary was found. "
        "RSAM processing may proceed, but SDS coverage has not been audited."
    )
```